<a href="https://colab.research.google.com/github/javisagredo-dev/Evaluacion1_Prog_DS/blob/main/Evaluaci%C3%B3n_1_Prog_DS.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import pandas as pd
import geopandas as gpd
from google.colab import drive
from shapely.geometry import Point

In [52]:
# Montar Google Drive
drive.mount('/content/drive')

# ============================================
# 1. Cargar sismos
# ============================================
df = pd.read_csv(
    '/content/drive/MyDrive/Ciencia de Datos Collab/Programación para la ciencia de datos/Evaluacion_1/Raw/archive.zip',
    compression='zip'
)

print(f"Total de sismos antes de filtrar: {len(df):,}")
#print(f"Columnas: {df.columns.tolist()}")
df.head(3)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Total de sismos antes de filtrar: 1,054,683


,id,time,year,month,day_of_year,hour,latitude,longitude,depth,mag,...,tsunami,mag_category,depth_category,nst,gap,dmin,rms,net,updated,status
0,cent19000105190000000,1900-01-05 19:00:00+00:00,1900,1,5,19,-3.0,102.0,NaN,7.0,...,0,Major (6-7),NaN,NaN,NaN,NaN,NaN,cent,2025-04-18T00:16:56.005Z,reviewed
1,cent19000111090700000,1900-01-11 09:07:00+00:00,1900,1,11,9,-5.0,148.0,NaN,7.0,...,0,Major (6-7),NaN,NaN,NaN,NaN,NaN,cent,2025-04-18T22:55:15.621Z,reviewed
2,cent19000120063300000,1900-01-20 06:33:00+00:00,1900,1,20,6,20.0,-105.0,NaN,7.3,...,0,Great (7-8),NaN,NaN,NaN,NaN,NaN,cent,2025-04-19T23:36:34.400Z,reviewed


In [53]:
# ============================================
# 2. Cargar polígonos de Chile (tierra y mar)
# ============================================
print("\nCargando polígonos...")

# Tierra: Regiones de Chile
tierra = gpd.read_file(
    '/content/drive/MyDrive/Ciencia de Datos Collab/Programación para la ciencia de datos/Evaluacion_1/Raw/REGIONES_v1.shp'
)

# Mar: ZEE de Chile (desde el XML)
mar = gpd.read_file(
    '/content/drive/MyDrive/Ciencia de Datos Collab/Programación para la ciencia de datos/Evaluacion_1/Raw/eez.xml'
)

# ============================================
# CORRECCIÓN: Asignar CRS si no tiene
# ============================================
if tierra.crs is None:
    print("Asignando CRS a REGIONES (EPSG:4326)...")
    tierra = tierra.set_crs("EPSG:4326")

if mar.crs is None:
    print("Asignando CRS a ZEE (EPSG:4326)...")
    mar = mar.set_crs("EPSG:4326")

# Asegurar mismo sistema de coordenadas (WGS84 = EPSG:4326)
tierra = tierra.to_crs("EPSG:4326")
mar = mar.to_crs("EPSG:4326")

print(f"CRS de tierra: {tierra.crs}")
print(f"CRS de mar: {mar.crs}")

# Unir tierra + mar en un solo polígono
chile_total = pd.concat([tierra, mar], ignore_index=True)
chile_total = gpd.GeoDataFrame(chile_total, geometry='geometry', crs="EPSG:4326")
chile_total = chile_total.dissolve()  # Fusiona todo en un solo polígono

print("Polígonos cargados y unificados correctamente")



Cargando polígonos...
Asignando CRS a REGIONES (EPSG:4326)...
Asignando CRS a ZEE (EPSG:4326)...
CRS de tierra: EPSG:4326
CRS de mar: EPSG:4326
Polígonos cargados y unificados correctamente


In [54]:
# ============================================
# 3. Convertir sismos a GeoDataFrame
# ============================================
print("\nCreando geometría de puntos...")

sismos_gdf = gpd.GeoDataFrame(
    df,
    geometry=gpd.points_from_xy(df['longitude'], df['latitude']),
    crs="EPSG:4326"
)

print(f"{len(sismos_gdf):,} puntos creados")


Creando geometría de puntos...
1,054,683 puntos creados


In [55]:
# ============================================
# 4. Filtrar sismos dentro de Chile
# ============================================
print("\nFiltrando sismos dentro de Chile...")

# Usamos intersects() en lugar de within() para incluir bordes
sismos_chile = sismos_gdf[sismos_gdf.intersects(chile_total.geometry.iloc[0])]

print(f"\n{'='*50}")
print(f"RESULTADOS DEL FILTRADO:")
print(f" Sismos TOTALES: {len(df):,}")
print(f" Sismos en CHILE: {len(sismos_chile):,}")
print(f" Sismos FUERA: {len(df) - len(sismos_chile):,}")
print(f" Porcentaje en Chile: {(len(sismos_chile)/len(df))*100:.2f}%")
print(f"{'='*50}")


Filtrando sismos dentro de Chile...

RESULTADOS DEL FILTRADO:
 Sismos TOTALES: 1,054,683
 Sismos en CHILE: 39,616
 Sismos FUERA: 1,015,067
 Porcentaje en Chile: 3.76%


In [ ]:
import folium
from folium.plugins import HeatMap

# Asegurarte de que no haya nulos en lat/lon
sismos_chile = sismos_chile.dropna(subset=['latitude', 'longitude'])

# Crear mapa centrado en Chile
mapa_chile = folium.Map(
    location=[-35.5, -71.0],
    zoom_start=4,
    tiles='CartoDB positron'
)

# Extraer coordenadas para el heatmap
heat_data = sismos_chile[['latitude', 'longitude']].values.tolist()

# Agregar mapa de calor
HeatMap(
    heat_data,
    radius=8,
    blur=6,
    max_zoom=10
).add_to(mapa_chile)

# Mostrar mapa
mapa_chile

In [57]:
##EXPORTACION DEL DATAFRAME A PROCESSED
import os

ruta_output = "/content/drive/MyDrive/Ciencia de Datos Collab/Programación para la ciencia de datos/Evaluacion_1/Processed/sismos_chile.csv"
sismos_chile.to_csv(ruta_output, index=False)

print(f"Archivo CSV guardado en: {ruta_output}")

Archivo CSV guardado en: /content/drive/MyDrive/Ciencia de Datos Collab/Programación para la ciencia de datos/Evaluacion_1/Processed/sismos_chile.csv


In [76]:
sismos_procesado = sismos_chile.copy()

In [77]:
## ANALISIS EXPLORATORIO DE DATOS

#Nulos
print(sismos_procesado.isnull().sum())

# Valores únicos en la columna 'type'
print("Valores únicos en 'type':")
print(sismos_procesado['type'].unique())
print(f"Cantidad de valores únicos: {sismos_procesado['type'].nunique()}\n")

# Valores únicos en la columna 'tsunami'
print("Valores únicos en 'tsunami':")
print(sismos_procesado['tsunami'].unique())
print(f"Cantidad de valores únicos: {sismos_procesado['tsunami'].nunique()}")

## DADO QUE LAS COLUMNAS TYPE Y TSUNAMI NO TIENEN VALORES DISTINTOS NO NOS SIRVEN PARA REALIZAR UN ESTUDIO ASI QUE LOS VAMOS A ELIMINAR

## nst, gap, dmin, rms: Son columnas con un alto porcentaje de valores nulos (por ejemplo, dmin tiene ~80% de nulos). Estas métricas son útiles para evaluar la
##  calidad de la localización del sismo desde el punto de vista instrumental, pero para nuestro objetivo no son relevantes. Además, imputar valores tan
##  faltantes introduciría ruido innecesario.

## net, updated y status: net indica la red sísmica que reportó el evento (no aporta a nuestro modelo), updated es una marca de tiempo de la última actualización del registro,
##    no un atributo físico del sismo y finalmente status Indica si el evento fue revisado automática o manualmente lo cual no aporta a nuestro estudio.

## geometry: Es la columna de geometría del GeoDataFrame. Como nuestro análisis posterior no requiere operaciones espaciales (o porque ya extrajimos latitud/longitud),
##    la eliminamos para reducir memoria y simplificar el DataFrame. Si necesitamos mapas, podemos reconstruirla después.

## TODAS LAS COLUMNAS AQUI EVALUADAS SON CANDIDATAS A SER ELIMINADAS DE LA MUESTRA YA QUE NO NOS SIRVEN PARA NUESTRO ESTUDIO, ESTO NOS PERMITIRÁ DEJAR EL DATASET MAS LIMPIO
## ADEMAS SE SUMAN A ESTA LISTA LAS COLUMNAS time, year, month, day of the year y hour, ya que estas seran agrupadas en 2 unicas columnas de fecha y hora



id                    0
time                  0
year                  0
month                 0
day_of_year           0
hour                  0
latitude              0
longitude             0
depth                 0
mag                   0
magType               0
place                 0
type                  0
tsunami               0
mag_category          0
depth_category        0
nst               18872
gap               17646
dmin              31627
rms               18776
net                   0
updated               0
status                0
geometry              0
dtype: int64
Valores únicos en 'type':
['earthquake']
Cantidad de valores únicos: 1

Valores únicos en 'tsunami':
[0]
Cantidad de valores únicos: 1


In [78]:
## INICIO DEL PROCEDIMIENTO

sismos_procesado['fecha'] = pd.to_datetime(
    sismos_procesado['year'].astype(str) + '-' + sismos_procesado['day_of_year'].astype(str),
    format='%Y-%j')
sismos_procesado['hora'] = sismos_procesado['hour']

## Mascara de columnas para eliminar
columnas_a_eliminar = [
    'time', 'year', 'month', 'day_of_year', 'hour',   # originales de tiempo
    'nst', 'gap', 'dmin', 'rms', 'net', 'updated',    # nulos y metadata
    'status', 'geometry', 'type', 'tsunami'           # otras que no sirven para el estudio
]

## Aqui verificamos que las columnas existen antes de eliminar
existentes = [col for col in columnas_a_eliminar if col in sismos_procesado.columns]
sismos_procesado.drop(columns=existentes, inplace=True)

sismos_procesado.info()

<class 'geopandas.geodataframe.GeoDataFrame'>
Index: 39616 entries, 83 to 1054663
Data columns (total 11 columns):
 #   Column          Non-Null Count  Dtype         
---  ------          --------------  -----         
 0   id              39616 non-null  object        
 1   latitude        39616 non-null  float64       
 2   longitude       39616 non-null  float64       
 3   depth           39616 non-null  float64       
 4   mag             39616 non-null  float64       
 5   magType         39616 non-null  object        
 6   place           39616 non-null  object        
 7   mag_category    39616 non-null  object        
 8   depth_category  39616 non-null  object        
 9   fecha           39616 non-null  datetime64[ns]
 10  hora            39616 non-null  int64         
dtypes: datetime64[ns](1), float64(4), int64(1), object(5)
memory usage: 3.6+ MB


In [ ]:
# Identificar outliers
Q1 = sismos_procesado['mag'].quantile(0.25)
Q3 = sismos_procesado['mag'].quantile(0.75)
IQR = Q3 - Q1
limite_inferior = Q1 - 1.5 * IQR
limite_superior = Q3 + 1.5 * IQR

outliers_mag = sismos_procesado[(sismos_procesado['mag'] < limite_inferior) |
                                 (sismos_procesado['mag'] > limite_superior)]
print(f"Magnitudes fuera de rango IQR: {len(outliers_mag)} eventos")
print("Estos corresponden a sismos de magnitud > ~6.5, que son los más relevantes para tsunamis. Por lo tanto, se conservan.")

## A partir de el analisis de los outliers tomamos las siguientes decisiones

Decisión 1: Preservación de valores extremos (outliers)

En el proceso de limpieza, se identificaron valores atípicos en columnas como depth (profundidad) y mag (magnitud). Por ejemplo, profundidades negativas (errores instrumentales) o superiores a 600 km, y magnitudes mayores a 8.5. La práctica habitual en ciencia de datos sería recortar o eliminar estos registros. Sin embargo, en el contexto del estudio de sismicidad y riesgo de tsunamis, los valores extremos son los más relevantes desde el punto de vista científico.
Razones:
- Los terremotos de gran magnitud (Mw > 7.5) son poco frecuentes pero concentran la mayor parte del potencial destructivo y de generación de tsunamis. Eliminarlos como outliers sesgaría completamente el análisis, eliminando los eventos más importantes para la predicción de catástrofes.
- Profundidades anómalas (muy someras o muy profundas) permiten estudiar la correlación entre profundidad y generación de tsunamis. Por ejemplo, sismos superficiales (< 30 km) con alta magnitud son los que más tsunamis generan; sismos de subducción profunda (> 300 km) raramente generan tsunamis, pero su inclusión ayuda a delimitar umbrales.
- En lugar de eliminar outliers, se opta por mantener todos los registros originales y, en etapas posteriores (si se requiere modelado), se aplicarán técnicas robustas como RobustScaler (que usa mediana y rango intercuartil) o modelos basados en árboles que no son sensibles a valores extremos.

Por lo tanto, no se realiza ningún filtro por outliers en profundidad ni magnitud, preservando la integridad del fenómeno natural.

Decisión 2: Mantener la magnitud en su escala original sin estandarizar ni codificar

El dataset incluye una columna mag (valor numérico de magnitud) y una columna magType que indica la escala utilizada: ms (ondas superficiales), mw (magnitud momento), mb (ondas de cuerpo), ml (magnitud local), md (duración), mwb, mwc, mww, mwr (variantes de magnitud momento), y m (genérica). Cada una de estas escalas mide aspectos distintos del tamaño del sismo y no son linealmente equivalentes ni directamente comparables.

Razones para no aplicar técnicas de estandarización o codificación sobre la magnitud:

- Estandarizar (por ejemplo, con StandardScaler) implicaría restar la media y dividir por la desviación estándar de todos los valores de mag, independientemente de su magType. Esto mezcla escalas heterogéneas y produce una variable artificial sin interpretación física. Un sismo de mb=6.0 no es equivalente a uno de mw=6.0; la magnitud momento es más precisa y generalmente más alta para grandes terremotos.

- Aplicar codificación One-Hot a magType y multiplicar por mag tampoco resuelve el problema, porque la magnitud sigue siendo un número que proviene de diferentes algoritmos. Lo correcto sería convertir todas las magnitudes a una escala común (por ejemplo, a mw usando relaciones empíricas publicadas en sismología), pero eso excede el alcance de este proyecto y podría introducir errores sistemáticos.

- Estrategia adoptada: Se decide no transformar la columna mag numéricamente (no se escala, no se normaliza). En cambio, se conserva mag en su valor original y se utiliza magType como una variable categórica para análisis segmentados. Por ejemplo, se pueden generar visualizaciones separadas por tipo de magnitud, o filtrar solo eventos con magType='mw' (la más confiable) para ciertos modelos. Cualquier intento de estandarización global sería inapropiado y científicamente incorrecto.

Esta decisión mantiene la trazabilidad y el significado físico de los datos, evitando artefactos estadísticos sin respaldo geofísico.